# Imports

In [1]:
import time
from typing import Callable, Union, Union
import torch
import torch.nn.functional as F
from torch.optim import Optimizer, SGD
from torch.utils.data import DataLoader
from torch import Tensor
import argparse
import json
import tensorboard
import tensorboardX
import os
import argparse
import json
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim 
import nni
from nni.nas.nn.pytorch import ModelSpace, LayerChoice, MutableConv2d, MutableBatchNorm2d, MutableReLU
from pytorch_lightning import Trainer
from nni.nas.evaluator.pytorch import Lightning, ClassificationModule, Trainer
from nni.nas.experiment import NasExperiment
from nni.nas.space import model_context
from nni.nas.hub.pytorch import DARTS
from nni.nas.strategy import DARTS as DartsStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision import transforms
from torchvision.datasets import CIFAR10
from nni.nas.experiment import NasExperiment
from nni.nas.evaluator import FunctionalEvaluator
from nni.nas.evaluator import FunctionalEvaluator
import nni.nas.strategy as strategy
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
#from ops import AvgPool,DilConv,SepConv
import genotypes
from pytorch_lightning.callbacks import ModelCheckpoint
torch.set_float32_matmul_precision('medium')
from tqdm import tqdm
from nni.nas.nn.pytorch import LayerChoice, ModelSpace,ValueChoice
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from pytorch_lightning import LightningModule, Trainer
from torchvision import datasets, transforms
from nni.nas.evaluator.pytorch import Classification
from nni.common.types import SCHEDULER
import nni
from nni.compression.quantization import QATQuantizer
from nni.compression.utils import TorchEvaluator
from torch.nn import utils
import torch.nn.utils as nn_utils
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
import time


# Model

### DARTS found architecture

PhotonicArch(
  (layer0_conv): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), bias=False)
  (layer0_bn): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer0_relu): ReLU()
  (layer1_avgpool): AvgPool2d(kernel_size=3, stride=1, padding=0)
  (layer1_conv): Conv2d(8, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (layer1_bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1_relu): ReLU()
  (layer2_avgpool): AvgPool2d(kernel_size=3, stride=1, padding=0)
  (layer2_conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (layer2_bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer2_relu): ReLU()
  (layer3_conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (layer3_avgpool): AvgPool2d(kernel_size=3, stride=1, padding=0)
  (layer3_bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer3_relu): ReLU()
  (layer4_conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (layer4_avgpool): AvgPool2d(kernel_size=3, stride=1, padding=0)
  (layer4_bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer4_relu): ReLU()
  (layer5_avgpool): AvgPool2d(kernel_size=3, stride=1, padding=0)
  (layer5_conv): Conv2d(64, 22, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (layer5_bn): BatchNorm2d(22, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer5_relu): ReLU()
  (pool): AdaptiveAvgPool2d(output_size=(3, 3))
  (fc1): Linear(in_features=198, out_features=160, bias=True)
  (fc2): Linear(in_features=160, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=96, bias=True)
  (relu): ReLU()
  (classifier): Linear(in_features=96, out_features=10, bias=True)


### Found architecture implementation from scratch

In [15]:
class PhotonicArch(torch.nn.Module):
    def __init__(self, drop_path_prob=0.0):
        super().__init__()
        
        self.drop_path_prob = drop_path_prob 
        #________________________________________________________________________________________________________________________
        #Layer 0
        self.layer0_conv = torch.nn.Conv2d(3, 8, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(8)
        self.relu=torch.nn.ReLU(inplace=False)
        #________________________________________________________________________________________________________________________
        #Layer 1
        self.layer1_avgpool= torch.nn.AvgPool2d(kernel_size=2, stride=1, padding=0)
        self.layer1_conv=torch.nn.Conv2d(8, 32, kernel_size=3, stride=1, padding=1)
        self.layer1_bn=torch.nn.BatchNorm2d(32, affine=True)
        #________________________________________________________________________________________________________________________
        #Layer 2 
        self.layer2_avgpool= torch.nn.AvgPool2d(kernel_size=2, stride=1, padding=0)
        self.layer2_conv=torch.nn.Conv2d(32, 16, kernel_size=3, stride=1, padding=1)
        self.layer2_bn=torch.nn.BatchNorm2d(16, affine=True)
        #________________________________________________________________________________________________________________________
        #Layer 3
        self.layer3_conv=torch.nn.Conv2d(16, 64, kernel_size=3, stride=1, padding=1)
        self.layer3_avgpool= torch.nn.AvgPool2d(kernel_size=2, stride=1, padding=0)
        self.layer3_bn=torch.nn.BatchNorm2d(64, affine=True)
        #________________________________________________________________________________________________________________________
        #Layer 4
        self.layer4_conv=torch.nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)
        self.layer4_avgpool= torch.nn.AvgPool2d(kernel_size=2, stride=1, padding=0)
        self.layer4_bn=torch.nn.BatchNorm2d(64, affine=True)
        #________________________________________________________________________________________________________________________
        #Layer 5
        self.layer5_avgpool= torch.nn.AvgPool2d(kernel_size=2, stride=1, padding=0)
        self.layer5_conv=torch.nn.Conv2d(64, 16, kernel_size=3, stride=1, padding=1)
        self.layer5_bn=torch.nn.BatchNorm2d(16, affine=True)
        #________________________________________________________________________________________________________________________
        #Layer 6
        self.layer6_avgpool= torch.nn.AvgPool2d(kernel_size=2, stride=1, padding=0)
        self.layer6_conv=torch.nn.Conv2d(16, 16, kernel_size=3, stride=1, padding=1)
        self.layer6_bn=torch.nn.BatchNorm2d(16, affine=True)
        #________________________________________________________________________________________________________________________
        #Layer 7
        self.layer7_avgpool= torch.nn.AvgPool2d(kernel_size=2, stride=1, padding=0)
        self.layer7_conv=torch.nn.Conv2d(16, 22, kernel_size=3, stride=1, padding=1)
        self.layer7_bn=torch.nn.BatchNorm2d(22, affine=True)
        #________________________________________________________________________________________________________________________   
        self.pool = torch.nn.AdaptiveAvgPool2d((3, 3))
        self.fc1 = torch.nn.Linear(198, 32) 
        self.fc2 = torch.nn.Linear(32, 32) 
        self.fc3 = torch.nn.Linear(32, 32)  
        self.classifier = torch.nn.Linear(32, 10)

    def forward(self, x):
        #________________________________________________________________________________________________________________________
        x = self.layer0_conv(x)
        X = self.layer0_bn(x)
        x = self.relu(x)
        #________________________________________________________________________________________________________________________
        # Unroll layer1
        x = self.layer1_conv(x)
        x = self.layer1_avgpool(x)
        x = self.layer1_bn(x)
        x = self.relu(x)
        #print(f'After l1: {x.shape}')
        #________________________________________________________________________________________________________________________
        # Unroll layer2
        x = self.layer2_conv(x)
        x = self.layer2_avgpool(x)
        x = self.layer2_bn(x)
        x = self.relu(x)
        #print(f'After l2: {x.shape}')

        x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
        #________________________________________________________________________________________________________________________
        # Unroll layer3
        x = self.layer3_avgpool(x)
        x = self.layer3_conv(x)
        x = self.layer3_bn(x)
        x = self.relu(x)
        #print(f'After l3: {x.shape}')

        #________________________________________________________________________________________________________________________
        # Unroll layer4
        x = self.layer4_avgpool(x)
        x = self.layer4_conv(x)
        x = self.layer4_bn(x)
        x = self.relu(x)
        #print(f'After l4: {x.shape}')    
        x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
        #________________________________________________________________________________________________________________________
        # Unroll layer5
        x = self.layer5_avgpool(x)
        x = self.layer5_conv(x)
        x = self.layer5_bn(x)
        x = self.relu(x)
        #print(f'After l5: {x.shape}')    

        #________________________________________________________________________________________________________________________
        # Unroll layer6
        x = self.layer6_conv(x)
        x = self.layer6_avgpool(x)
        x = self.layer6_bn(x)
        x = self.relu(x)
        #print(f'After l5: {x.shape}')  
        #________________________________________________________________________________________________________________________
        # Unroll layer7
        x = self.layer7_conv(x)
        x = self.layer7_avgpool(x)
        x = self.layer7_bn(x)
        x = self.relu(x)
        #print(f'After l5: {x.shape}')  
        x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
        #________________________________________________________________________________________________________________________
        x =  self.pool(x)
        #print(f'After adaptive: {x.shape}')
        #________________________________________________________________________________________________________________________
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x= self.relu(x)
        x = self.fc2(x)
        x= self.relu(x)
        x = self.fc3(x)
        x= self.relu(x)
        
        x = self.classifier(x)
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob

# Dataloaders

In [13]:
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10

CIFAR10(root='data/cifar10', train=True, download=True)
CIFAR10(root='data/cifar10', train=False, download=True)


transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

cifar10_train = CIFAR10(root='data/cifar10', train=True, transform=transform)
train_dataloader = DataLoader(cifar10_train, batch_size=32, shuffle=True)

cifar10_test = CIFAR10(root='data/cifar10', train=False, transform=transform)
test_dataloader = DataLoader(cifar10_test, batch_size=1000, shuffle=False)


Files already downloaded and verified
Files already downloaded and verified


# Base Training and Evaluation

### Training Model

In [19]:
def training_model(model, optimizer, train_dataloader, device, max_epochs=50, save_path="best_model.pth", bestloss = float('inf'),eval_all = False, quant_eva = False, bestacc = 0.):
    #Set model into training mode
    model.train()
    criterion = nn.CrossEntropyLoss()
    #Initialize best loss to max value 
    best_loss = bestloss
    best_acc = bestacc
    
    for epoch in range(max_epochs):
        epoch_best_loss = float('inf')
        print(f"Epoch {epoch + 1}/{max_epochs} starts")
        
        #Iterate train dataloader
        for batch_idx, batch in enumerate(train_dataloader):
            
            #Reset grad for curr batch
            optimizer.zero_grad()
            
            #Get loss from training step
            loss = training_step(batch, model, device, criterion)
            
            #Propagate loss
            loss.backward()
            
            #Optimizer next step
            optimizer.step()
            if loss.item() < epoch_best_loss:
                epoch_best_loss=loss.item()
            #Check if best loss and save model
            #if loss.item()< 0.25:
        if epoch_best_loss < best_loss:
            best_loss = epoch_best_loss
            acc = evaluating_model(photonic_model, test_dataloader, device)
            print(f'Quantization evaluation: loss: {loss.item()}  Acc.: {acc}')
            if acc > best_acc:                     
                best_acc = acc
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),  # Save optimizer state
                    'best_acc': best_acc,  # Save best accuracy
                    'best_loss': best_loss  # Save best loss
                }, save_path)
                print(f"New best model saved with loss {best_loss:.4f} and acc {acc}")
                        
        print(f"Epoch {epoch + 1} Loss: {epoch_best_loss}")

        
    print(f"Training completed. Best model saved with loss {best_loss:.4f}")

### Training step

In [7]:
def training_step(batch, model, device, criterion, l1_coeff=1e-5):
    
    # Get data and labels
    x, y = batch
    
    # Put data and labels on GPU
    x, y = x.to(device), y.to(device)
    
    # Predict
    y_hat = model(x)
    
    # Calculate loss (cross entropy)
    loss_main = criterion(y_hat, y)
    
    #L1 regularization
    l1_regularization = sum(p.abs().sum() for p in model.parameters())
    loss = loss_main + (l1_coeff * l1_regularization)
    
    return loss_main

### Evaluate function

In [8]:
def evaluating_model(model, dataloader, device):
    # Set model in evaluation mode
    model.eval()
    
    correct, total = 0, 0
    
    with torch.no_grad():

        #Iterate validation dataloaders
        for x, y in dataloader:

            #Put data and labels on GPU
            x, y = x.to(device), y.to(device)

            #Get prediction from stoftmax argmax
            output = model(x)
            preds = torch.argmax(output, dim=1)

            #Increment counters
            correct += (preds == y).sum().item()
            total += y.size(0)

    #Return accuracy
    return correct / total

### Training

In [19]:
#Define cuda device fo GPU use
device = "cuda:0" if torch.cuda.is_available() else "cpu"

#Instantiate model and put on GPU
photonic_model = PhotonicArch().to(device)

#Initialize lr wd optimizer and scheduler
learning_rate = 1e-5
weight_decay = 0.
optimizer = Adam(photonic_model.parameters(), lr=1e-5, weight_decay=0.)

# Start training
start = time.time()
training_model(
    model=photonic_model,
    optimizer=optimizer,
    train_dataloader=train_dataloader,  
    device=device,
    max_epochs=200
)
print(f"Training completed in {time.time() - start:.2f}s")


Epoch 1/200 starts
New best model saved with loss 2.3270
New best model saved with loss 2.3153
New best model saved with loss 2.2606
New best model saved with loss 2.2581
New best model saved with loss 2.2434
New best model saved with loss 2.2361


KeyboardInterrupt: 

### Evaluation

In [5]:
#Instantiate model and put on GPU

device = "cuda:0" if torch.cuda.is_available() else "cpu"
photonic_model = PhotonicArch().to(device)

#Load weights from best model found in base training
photonic_model.load_state_dict(torch.load("best_model.pth"))

RuntimeError: Error(s) in loading state_dict for PhotonicArch:
	Missing key(s) in state_dict: "layer0_conv.weight", "layer0_bn.weight", "layer0_bn.bias", "layer0_bn.running_mean", "layer0_bn.running_var", "layer1_conv.weight", "layer1_conv.bias", "layer1_bn.weight", "layer1_bn.bias", "layer1_bn.running_mean", "layer1_bn.running_var", "layer2_conv.weight", "layer2_conv.bias", "layer2_bn.weight", "layer2_bn.bias", "layer2_bn.running_mean", "layer2_bn.running_var", "layer3_conv.weight", "layer3_conv.bias", "layer3_bn.weight", "layer3_bn.bias", "layer3_bn.running_mean", "layer3_bn.running_var", "layer4_conv.weight", "layer4_conv.bias", "layer4_bn.weight", "layer4_bn.bias", "layer4_bn.running_mean", "layer4_bn.running_var", "layer5_conv.weight", "layer5_conv.bias", "layer5_bn.weight", "layer5_bn.bias", "layer5_bn.running_mean", "layer5_bn.running_var", "fc1.weight", "fc1.bias", "fc2.weight", "fc2.bias", "fc3.weight", "fc3.bias", "classifier.weight", "classifier.bias". 
	Unexpected key(s) in

In [21]:
start = time.time()
acc = evaluating_model(photonic_model, test_dataloader, device)  
print(f'Pure evaluation: {time.time() - start}s    Acc.: {acc}')

Pure evaluation: 3.1331613063812256s    Acc.: 0.6928


# Quantized Training and Evaluation

### QATQuantizer

In [32]:
#Define cuda device fo GPU use
device = "cuda:0" if torch.cuda.is_available() else "cpu"

#Instantiate model and put on GPU
photonic_model = PhotonicArch().to(device)

#Load weights from best model found in base training
#photonic_model.load_state_dict(torch.load("best_model.pth"))

#Define optimizer and scheduler, optimizer wrapped with nni.trace since it gave issues without wrapping using a torch evaluator, god knows why
optimizer = nni.trace(Adam)(photonic_model.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=200, eta_min=1e-6)
trainingModel = training_model(
    model=photonic_model,
    optimizer=optimizer,
    train_dataloader=train_dataloader,  
    device=device,
    max_epochs=200, save_path="best_QAT_quantized_4bit_model.pth",
    quant_eva = True
)

# Initialize evaluator
evaluator = TorchEvaluator(
    trainingModel    , 
    optimizer, 
    training_step
)
# Quantizer configuration 
config_list = [{
    'op_names': ['layer0_conv','layer1_conv','layer2_conv','layer3_conv','layer4_conv','layer5_conv','layer6_conv','layer7_conv', 'fc1', 'fc2','fc3'],
    'target_names': ['_input_', 'weight', '_output_'],
    'quant_dtype': 'int4',
    'quant_scheme': 'affine',
    'granularity': 'default',
}]

#Initialize quantizer
quantizer = QATQuantizer(photonic_model, config_list, evaluator, len(train_dataloader))

# Calibrate the quantizer
real_input = next(iter(train_dataloader))[0].to(device)
quantizer.track_forward(real_input)

# Start quantization training
start = time.time()
_, calibration_config = quantizer.compress(None, max_epochs=200)
print(f'Quantization training for 600 epochs took: {time.time() - start}s')

# Start quantization evaluation
start = time.time()
acc = evaluating_model(photonic_model, test_dataloader, device)  # Assuming evaluating_model uses test_loader
print(f'Quantization evaluation: {time.time() - start}s    Acc.: {acc}')

Epoch 1/200 starts
Quantization evaluation: loss: 1.8829600811004639  Acc.: 0.4274
New best model saved with loss 1.1674 and acc 0.4274
Epoch 1 Loss: 1.1674108505249023
Epoch 2/200 starts
Quantization evaluation: loss: 1.1675050258636475  Acc.: 0.5126
New best model saved with loss 0.8558 and acc 0.5126
Epoch 2 Loss: 0.8558007478713989
Epoch 3/200 starts
Quantization evaluation: loss: 1.7584614753723145  Acc.: 0.5221
New best model saved with loss 0.7341 and acc 0.5221
Epoch 3 Loss: 0.7341431379318237
Epoch 4/200 starts
Quantization evaluation: loss: 1.5015678405761719  Acc.: 0.5928
New best model saved with loss 0.6476 and acc 0.5928
Epoch 4 Loss: 0.6475616693496704
Epoch 5/200 starts
Quantization evaluation: loss: 1.6975815296173096  Acc.: 0.6194
New best model saved with loss 0.5371 and acc 0.6194
Epoch 5 Loss: 0.5371088981628418
Epoch 6/200 starts
Quantization evaluation: loss: 1.0830470323562622  Acc.: 0.637
New best model saved with loss 0.4195 and acc 0.637
Epoch 6 Loss: 0.41947

TypeError: 'NoneType' object is not callable

### DoReFa

In [24]:

from nni.compression.quantization import DoReFaQuantizer
#Define cuda device fo GPU use
device = "cuda:0" if torch.cuda.is_available() else "cpu"

#Instantiate model and put on GPU
photonic_model = PhotonicArch().to(device)

me = 100

#Define optimizer and scheduler, optimizer wrapped with nni.trace since it gave issues without wrapping using a torch evaluator, god knows why
optimizer = nni.trace(Adam)(photonic_model.parameters(), lr=1e-3, weight_decay=0.)
#optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
trainingModel = training_model(
    model=photonic_model,
    optimizer=optimizer,
    train_dataloader=train_dataloader,  
    device=device,
    max_epochs=100, save_path="best_DoReFa_quantized_b8it_model.pth",
    quant_eva = True
)

evaluator = TorchEvaluator(
    trainingModel    , 
    optimizer, 
    training_step
)


# Quantizer configuration 
config_list = [{
    'op_names': ['layer0_conv','layer1_conv','layer2_conv','layer3_conv','layer4_conv','layer5_conv','layer6_conv','layer7_conv', 'fc1', 'fc2','fc3'],
    'target_names': ['_input_', 'weight'],
    'quant_dtype': 'int8',
    'quant_scheme': 'affine',
    'granularity': 'default',
}]

#Initialize quantizer
quantizer = DoReFaQuantizer(photonic_model, config_list, evaluator)

# Calibrate the quantizer
real_input = next(iter(train_dataloader))[0].to(device)
quantizer.track_forward(real_input)

# Start quantization training
start = time.time()
_, calibration_config = quantizer.compress(None, max_epochs=me)
print(f'Quantization training for {me} epochs took: {time.time() - start}s')

# Start quantization evaluation
acc = evaluating_model(photonic_model, test_dataloader, device)
print(f'Quantization evaluation:  Acc.: {acc}')

Epoch 1/100 starts
Quantization evaluation: loss: 1.961377501487732  Acc.: 0.3763
New best model saved with loss 1.2871 and acc 0.3763
Epoch 1 Loss: 1.287105679512024
Epoch 2/100 starts


KeyboardInterrupt: 

## Continue training


In [25]:
checkpoint = torch.load("best_DoReFa_quantized_b8it_model.pth")

In [26]:
transform = transforms.Compose([

    transforms.ToTensor()
])

cifar10_train = CIFAR10(root='data/cifar10', train=True, transform=transform)
train_dataloader = DataLoader(cifar10_train, batch_size=32, shuffle=True)

cifar10_test = CIFAR10(root='data/cifar10', train=False, transform=transform)
test_dataloader = DataLoader(cifar10_test, batch_size=1000, shuffle=False)

In [27]:
checkpoint['optimizer_state_dict']['param_groups'][0]['lr'] = 0.00005
checkpoint['optimizer_state_dict']['param_groups'][0]['lr'] 

5e-05

In [28]:

from nni.compression.quantization import DoReFaQuantizer
#Define cuda device fo GPU use
device = "cuda:0" if torch.cuda.is_available() else "cpu"
#Instantiate model and put on GPU
photonic_model = PhotonicArch().to(device)
photonic_model.load_state_dict(checkpoint['model_state_dict'])

#Define optimizer and scheduler, optimizer wrapped with nni.trace since it gave issues without wrapping using a torch evaluator, god knows why
optimizer = nni.trace(Adam)(photonic_model.parameters(), lr=1e-5, weight_decay=0.)
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])  
best_acc = checkpoint['best_acc']  
best_loss = checkpoint['best_loss']

print(f"loading model with acc : {best_acc}")

me = 500


trainingModel = training_model(
    model=photonic_model,
    optimizer=optimizer,
    train_dataloader=train_dataloader,  
    device=device,
    max_epochs=200, save_path="best_DoReFa_quantized_8bit_model.pth",
    bestacc = best_acc,
    quant_eva=True,
    eval_all = True
)

evaluator = TorchEvaluator(
    trainingModel    , 
    optimizer, 
    training_step
)


# Quantizer configuration 
config_list = [{
    'op_names': ['layer0_conv','layer1_conv','layer2_conv','layer3_conv','layer4_conv','layer5_conv', 'fc1', 'fc2','fc3'],
    'target_names': ['_input_', 'weight'],
    'quant_dtype': 'int8',
    'quant_scheme': 'affine',
    'granularity': 'default',
}]

#Initialize quantizer
quantizer = DoReFaQuantizer(photonic_model, config_list, evaluator)

# Calibrate the quantizer
real_input = next(iter(train_dataloader))[0].to(device)
quantizer.track_forward(real_input)

# Start quantization training
start = time.time()
_, calibration_config = quantizer.compress(None, max_epochs=me)
print(f'Quantization training for {me} epochs took: {time.time() - start}s')

# Start quantization evaluation
start = time.time()
acc = evaluating_model(photonic_model, test_dataloader, device)
print(f'Quantization evaluation: {time.time() - start}s    Acc.: {acc}')

loading model with acc : 0.3763
Epoch 1/200 starts
Quantization evaluation: loss: 1.314866542816162  Acc.: 0.4857
New best model saved with loss 1.0129 and acc 0.4857
Epoch 1 Loss: 1.0128542184829712
Epoch 2/200 starts


KeyboardInterrupt: 

In [64]:
checkpoint = torch.load("best_DoReFa_quantized_model.pth")

#Instantiate model and put on GPU
photonic_model = PhotonicArch().to(device)

#Load weights from best model found in base training
photonic_model.load_state_dict(checkpoint['model_state_dict'])
# Start quantization evaluation
start = time.time()
acc = evaluating_model(photonic_model, test_dataloader, device)
print(f'Quantization evaluation: {time.time() - start}s    Acc.: {acc}')

Quantization evaluation: 1.722869634628296s    Acc.: 0.8301
